## échauffement : construire une URL avec des paramètres

In [1]:
num = 3
type = "rue"
nom = "tourelles"
URL = f"https://api-adresse.data.gouv.fr/search/?q={num}+{type}+{nom},Paris&limit=1"

## échauffement : sélectionner des lignes avec iloc et loc

In [2]:
import pandas as pd

df = pd.DataFrame({
    "number": [3, 18, 7, 12, 5],
    "name": ["tourelles", "BERNARDINS", "Bouvart", "Rivoli", "Seine"],
})
sub = df[df.number > 10]

In [3]:
# Question 1

sub
# sub contient les adresses (nom de rue et numéro) dont le numéro est strictement supérieur à 10
# l'index reste le même que df (1 et 3)

,number,name
1,18,BERNARDINS
3,12,Rivoli


In [4]:
# Question 2

print(df.iloc[:2])
# garde les élément jusqu'à la position 2 exclue
print(df.loc[:2])
# garde les élément jusqu'à l'index 2 compris

   number        name
0       3   tourelles
1      18  BERNARDINS
   number        name
0       3   tourelles
1      18  BERNARDINS
2       7     Bouvart


In [5]:
# Question 3

print(sub.iloc[:2])
# garde les élément jusqu'à la position 2 exclue (dans ce cas, garde tout car 2 éléments)
print(sub.loc[:2])
# garde les élément jusqu'à l'index 2 compris (donc ne prend pas rivoli qui était indéxé par 3 dans le tableau d'origine)

   number        name
1      18  BERNARDINS
3      12      Rivoli
   number        name
1      18  BERNARDINS


In [6]:
# Question 4

print(sub.iloc[0])
# renvoie le premier élément de la dataframe
#print(sub.loc[0])
# renvoie l'élément indéxé par 0 (or aucun dans sub donc renvoie une erreur)

number            18
name      BERNARDINS
Name: 1, dtype: object


In [7]:
# Question 5

print(sub.iloc[0])

print(sub.loc[min(sub.index)]) 
# On récupère l'index minimal 


number            18
name      BERNARDINS
Name: 1, dtype: object
number            18
name      BERNARDINS
Name: 1, dtype: object


## géolocalisation

In [8]:
# for starters, we only need the 'regular fit' pandas

import pandas as pd

In [9]:
# load the data in data/addresses.csv
# and display a few first lines

addresses = pd.read_csv('data/addresses.csv')
addresses.head(4)

,number,type,name,city
0,5,rue,RIQUET,Paris
1,141,boulevard,ITALIE,Paris
2,234,rue,FAUBOURG ST-HONORÉ,Paris
3,62,rue,MICHEL BIZOT,Paris


In [10]:
# produce a small extract into data/addresses-small.csv

# your code here
addresses.iloc[:20].to_csv('data/addresses-small.csv', index=False)

### une par une 

In [11]:
# requests is the swiss knife for doing http

import requests

In [12]:
# votre code

def localize_one(num, type, nom):
    URL = f"https://api-adresse.data.gouv.fr/search/?q={num}+{type}+{nom},Paris&limit=1"
    r = requests.get(URL)
    if r.status_code<200 or r.status_code>=300 :
        return None
    else :
        return r.json()

In [13]:
# and here is how we could use it

localize_one(18, 'rue', 'BERNARDINS')

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [2.350728, 48.850345]},
   'properties': {'label': '18 Rue des Bernardins 75005 Paris',
    'score': 0.7544495104895104,
    'housenumber': '18',
    'id': '75105_0896_00018',
    'banId': 'acbebc10-eac0-475f-9466-e7dbe9446b02',
    'name': '18 Rue des Bernardins',
    'postcode': '75005',
    'citycode': '75105',
    'x': 652355.3,
    'y': 6861340.65,
    'city': 'Paris',
    'district': 'Paris 5e Arrondissement',
    'context': '75, Paris, Île-de-France',
    'type': 'housenumber',
    'importance': 0.68356,
    'depcode': '75',
    'street': 'Rue des Bernardins',
    '_type': 'address'}}],
 'query': '18 rue BERNARDINS,Paris'}

In [14]:
# try to estimate how long it would take 
# to resolve 20_000 addresses this way

import time

t1 = time.time()
localize_one(18, 'rue', 'BERNARDINS')
t2 = time.time()
20000*(t2-t1)/60
# Il faudrait 40 minutes pour obtenir 20 000 adresses

35.79982121785482

### en un seul coup

In [15]:
# requests is the swiss knife for doing http

import requests
# this one is to fake a file from a string
import io

def mass_post(filename, col_number, col_type, col_name, col_city):

    # we don't care about the index for now
    # but note that if we want to merge the result
    # with the initial data, there is a need 
    # for a unique index...

    with open(filename) as feed:
        response = requests.post(
            "https://api-adresse.data.gouv.fr/search/csv/", 
            files={'data': feed},
            data={'columns': [col_number, col_type, col_name, col_city]})
    if not (200 <= response.status_code < 300):
        print(f"OOPS, got {response.status_code}")
        return None

    # this time the output is csv, not json
    # note that read_csv requires a file-like object
    # and we have a str, so io.StringIO comes to the rescue
    result = pd.read_csv(io.StringIO(response.text))

    return result



In [16]:
# votre code

def localize_many(filename, col_number, col_type, col_name, col_city):
    adresses = pd.read_csv(filename)
    geoloc = mass_post(filename, col_number, col_type, col_name, col_city).loc[:,['latitude', 'longitude', 'result_city', 'result_type']]
    result = pd.merge(adresses, geoloc, how='left', left_index=True, right_index=True)
    return result


In [17]:
# try your code on the small sample for starters

addresses_small = localize_many("data/addresses-small.csv", "number", "type", "name", "city")
addresses_small

,number,type,name,city,latitude,longitude,result_city,result_type
0,5,rue,RIQUET,Paris,48.888134,2.376448,Paris,housenumber
1,141,boulevard,ITALIE,Paris,46.669103,-1.418872,La Roche-sur-Yon,street
2,234,rue,FAUBOURG ST-HONORÉ,Paris,48.876160,2.302766,Paris,housenumber
3,62,rue,MICHEL BIZOT,Paris,48.835423,2.402040,Paris,housenumber
4,6,rue,OURS,Paris,48.863311,2.351775,Paris,housenumber
5,30,rue,PETIT-MUSC,Paris,48.852945,2.364722,Paris,housenumber
6,3,impasse,MASSONET,Paris,48.895498,2.351580,Paris,housenumber
7,56,rue,PANOYAUX,Paris,48.867023,2.388347,Paris,housenumber
8,16,rue,THÉRÈSE,Paris,48.866374,2.335212,Paris,housenumber
9,11,rue,JEAN ROBERT,Paris,48.889520,2.358170,Paris,housenumber


In [18]:
# sanity check : look at the entries that have
# result_city != 'Paris'
# (or result_type != 'housenumber')
# and decide what to do with them

def retirer_problemes(geoloc) :
    return geoloc[(geoloc.result_city.fillna("")=='Paris')&(geoloc.result_type.fillna("")=='housenumber')]

addresses_small = retirer_problemes(addresses_small)
addresses_small

,number,type,name,city,latitude,longitude,result_city,result_type
0,5,rue,RIQUET,Paris,48.888134,2.376448,Paris,housenumber
2,234,rue,FAUBOURG ST-HONORÉ,Paris,48.876160,2.302766,Paris,housenumber
3,62,rue,MICHEL BIZOT,Paris,48.835423,2.402040,Paris,housenumber
4,6,rue,OURS,Paris,48.863311,2.351775,Paris,housenumber
5,30,rue,PETIT-MUSC,Paris,48.852945,2.364722,Paris,housenumber
6,3,impasse,MASSONET,Paris,48.895498,2.351580,Paris,housenumber
7,56,rue,PANOYAUX,Paris,48.867023,2.388347,Paris,housenumber
8,16,rue,THÉRÈSE,Paris,48.866374,2.335212,Paris,housenumber
9,11,rue,JEAN ROBERT,Paris,48.889520,2.358170,Paris,housenumber
10,22,rue,FAUBOURG POISSONNIÈRE,Paris,48.872540,2.348002,Paris,housenumber


In [19]:
# when you think you're ready, go full scale
# be ready to wait for at least 40-60s 

# optional: try to record the time it takes !
# tip: see for example time.time()

t1 = time.time()
addresses = localize_many("data/addresses.csv", "number", "type", "name", "city")
addresses = retirer_problemes(addresses)
t2 = time.time()
print(t2-t1)
addresses


KeyboardInterrupt: 

## afficher sur une carte

In [20]:
import folium

In [21]:
# ~ chatelet

CENTER = 48.856542, 2.347614

### le fond de carte

In [22]:
# pour commencer on va recharger la dataframe précédente
import pandas as pd

addresses = pd.read_csv("data/addresses-geoloc.csv")

# et en faire un petit échantillon

addresses_small = addresses.iloc[:20]


In [23]:
# créez une map centrée sur ce point et de zoom 13
# n'oubliez pas de l'afficher,
# et vérifiez que vous voyez bien Paris, 
# que vous pouvez zoomer et vous déplacer, ...

# votre code
def paris_map():
    return folium.Map(location=CENTER, zoom_start=13, zoom_control=True)

# pour l'afficher
paris_map()


### on ajoute les adresses

In [24]:
# your code here

def map_addresses(geoloc):
    map = folium.Map(location=CENTER, zoom_start=13, zoom_control=True)
    for i in geoloc.index :
        ligne = geoloc.loc[i]
        folium.Marker(location=[ligne['latitude'], ligne['longitude']], tooltip=f"{ligne['number']} {ligne['type']} {ligne['name']}").add_to(map)
    return map

In [25]:
# and try it out

# make sure you use A SMALL DATAFRAME because with this method
# trying to display tens of thousands addresses
# is going to be SUPER SLOOOOOW !

map_addresses(addresses_small)

### on sauve la carte

In [27]:
map_small = map_addresses(addresses_small)
map_small.save("addresses-small.html")

## les quartiers de Paris

In [34]:
# à ce stade on a besoin de geopandas
# installez-le si besoin

import geopandas as gpd

# ça se lit facilement
quartiers = gpd.read_file("data/quartier_paris.zip", encoding="utf8")

# et le résultat est .. une dataframe
type(quartiers)

# qu'on peut afficher normalement

quartiers


AttributeError: partially initialized module 'pyproj' from 'c:\miniconda\Lib\site-packages\pyproj\__init__.py' has no attribute 'network' (most likely due to a circular import)